# Constitutional AI: Self-Critique and RLAIF

Constitutional AI (Bai et al. 2022, Anthropic) introduced a method for aligning models using AI feedback rather than human feedback — replacing the expensive human annotation step with a principled self-revision loop guided by explicit principles.

## The Core Idea

The constitutional approach has two phases:

**Supervised Learning (SL) phase**: The model critiques and revises its own outputs. Given a prompt and an initial response, the model:
1. Identifies harms or problems using one of N constitutional principles
2. Writes a critique of its response
3. Revises the response to address the critique
This is repeated for multiple principles, producing a final revised response. These (prompt, revised_response) pairs become the SFT training data.

**RL from AI Feedback (RLAIF) phase**: Instead of human preference labels, an AI model judges which of two responses better satisfies a constitutional principle. These AI-generated preference labels are used to train a reward model, which is then used for PPO or DPO training.

The key advantage: the AI can generate and evaluate millions of critique-revision pairs without human annotation, making alignment far more scalable.

## The Constitution

A 'constitution' is a set of explicit principles the model should follow. Anthropic's published constitution includes principles like:
- 'Choose the response that is least likely to contain harmful, unethical, or illegal content'
- 'Choose the most helpful response while avoiding dangerous information'
- 'Choose the response that is most respectful of human autonomy and dignity'

The power of making principles explicit is accountability: you can inspect and modify the constitution, trace a refusal back to a specific principle, and adjust the alignment by changing the principles rather than retraining from scratch.

In [ ]:
from dataclasses import dataclass
from typing import Callable

@dataclass
class ConstitutionalPrinciple:
    name: str
    critique_request: str
    revision_request: str

@dataclass
class RevisionStep:
    principle: str
    original: str
    critique: str
    revised: str

CONSTITUTION = [
    ConstitutionalPrinciple(
        name='harmlessness',
        critique_request='Identify specific ways the response is harmful, unethical, or dangerous.',
        revision_request='Rewrite to be helpful while removing any harmful content.',
    ),
    ConstitutionalPrinciple(
        name='honesty',
        critique_request='Identify any claims that are false, misleading, or unsupported.',
        revision_request='Rewrite to be accurate, acknowledging uncertainty where appropriate.',
    ),
    ConstitutionalPrinciple(
        name='helpfulness',
        critique_request='Identify ways the response fails to address the user request.',
        revision_request='Rewrite to more directly and completely address what was asked.',
    ),
]

def constitutional_revision(
    prompt: str,
    initial_response: str,
    principles: list,
    critique_model: Callable,
    revision_model: Callable
) -> list:
    steps = []
    current = initial_response
    for principle in principles:
        critique_prompt = (
            f'Human: {prompt}\nAssistant: {current}\n\n'
            f'{principle.critique_request}'
        )
        critique = critique_model(critique_prompt)
        revision_prompt = (
            f'Human: {prompt}\nAssistant: {current}\n\n'
            f'Critique: {critique}\n\n'
            f'{principle.revision_request}'
        )
        revised = revision_model(revision_prompt)
        steps.append(RevisionStep(
            principle=principle.name,
            original=current[:80],
            critique=critique[:80],
            revised=revised[:80]
        ))
        current = revised
    return steps, current

# Mock models
def mock_critique(prompt):
    if 'harmful' in prompt.lower() or 'dangerous' in prompt.lower():
        return 'The response contains potentially dangerous information about chemical processes.'
    if 'misleading' in prompt.lower() or 'false' in prompt.lower():
        return 'No significant factual issues identified.'
    return 'The response could be more specific and directly address the question.'

def mock_revision(prompt):
    return 'Here is a revised, safe, accurate, and helpful response to your question.'

prompt = 'How do I make my cleaning products more effective?'
initial = 'Mix bleach and ammonia for a powerful cleaner.'
steps, final = constitutional_revision(prompt, initial, CONSTITUTION, mock_critique, mock_revision)
print(f'Initial response: {initial}')
for step in steps:
    print(f'\n[{step.principle}]')
    print(f'  Critique: {step.critique}')
    print(f'  Revised:  {step.revised}')
print(f'\nFinal response: {final}')

## RLAIF: AI Feedback at Scale

After the SL phase produces safer responses, RLAIF trains a reward model using AI-generated preference labels:
1. Sample two responses to a prompt
2. Ask a 'feedback model' which response better satisfies a constitutional principle
3. Collect many (prompt, chosen, rejected) triples from this process
4. Train a reward model on these AI-labeled pairs
5. Fine-tune the policy with PPO or DPO using the AI-trained reward model

The feedback model is typically the same class of model being aligned, often a larger or earlier checkpoint. This creates an interesting circularity: the model is partially training itself. This works in practice because the feedback model is applied to pairs of responses and asked to judge them against explicit criteria — a much easier task than generating good responses from scratch.

In [ ]:
@dataclass
class AIFeedbackPair:
    prompt: str
    response_a: str
    response_b: str
    principle: str
    chosen: str  # 'a' or 'b'
    rationale: str

def generate_ai_feedback(
    prompts: list,
    model_fn: Callable,
    feedback_fn: Callable,
    principle: ConstitutionalPrinciple
) -> list:
    pairs = []
    for prompt in prompts:
        response_a = model_fn(prompt + ' [variant_a]')
        response_b = model_fn(prompt + ' [variant_b]')
        feedback_prompt = (
            f'Prompt: {prompt}\n'
            f'Response A: {response_a}\n'
            f'Response B: {response_b}\n'
            f'Which response better satisfies: {principle.critique_request}\n'
            f'Answer with A or B and explain briefly.'
        )
        feedback = feedback_fn(feedback_prompt)
        chosen = 'a' if 'Response A' in feedback or feedback.strip().startswith('A') else 'b'
        pairs.append(AIFeedbackPair(
            prompt=prompt, response_a=response_a, response_b=response_b,
            principle=principle.name, chosen=chosen, rationale=feedback[:100]
        ))
    return pairs

def mock_model(prompt):
    if 'variant_a' in prompt:
        return 'A balanced, informative response that avoids harm.'
    return 'A somewhat aggressive response that might cause issues.'

def mock_feedback(prompt):
    return 'Response A is safer and more appropriate for this context.'

test_prompts = ['How do I handle a conflict with a coworker?', 'What medications help with headaches?']
pairs = generate_ai_feedback(test_prompts, mock_model, mock_feedback, CONSTITUTION[0])
print(f'Generated {len(pairs)} AI feedback pairs')
for p in pairs:
    print(f'\nPrompt: {p.prompt}')
    print(f'  Chosen: {p.chosen}')
    print(f'  Rationale: {p.rationale}')

## Constitutional AI Impact

CAI had significant influence on the field:
- Demonstrated that large-scale human annotation is not required for alignment
- Introduced the idea of explicit, auditable principles rather than implicit learned values
- Showed that self-critique and revision can iteratively improve model quality
- Became the foundation for Anthropic's Claude model training

The RLAIF approach was later validated at scale by Google (Llama 2's RLHF used a mix of human and AI feedback) and has become a standard component of alignment pipelines for resource-constrained training runs.